# UNIVERSIDAD NACIONAL DE LOJA

## Documento Técnico: Pruebas de Hipótesis Unimuestrales y Comparación de Grupos

**Nombre:** Diyer Arley Torres Troya  

**Carrera:** Computación  

**Asignatura:** Teoría de la Distribución y Probabilidad  

**Docente:** Ing. Cristian Ramiro Narváez Guillén

In [11]:
from google.colab import files
uploaded = files.upload()

Saving 3._remuneraciones_ingresos_adicionales.xlsx to 3._remuneraciones_ingresos_adicionales (1).xlsx


In [12]:
!pip install openpyxl

**1. Pruebas de Hipótesis Unimuestrales (Test $t$)**

**Prueba de Hipótesis para la Remuneración Mensual Unificada**

Evaluaremos si la remuneración mensual promedio ($\mu$) de los servidores públicos en el Municipio de Loja difiere significativamente de un valor de referencia histórico nacional estimado en $\$900.00$. Dado que trabajamos con el dataset completo (o una muestra donde la varianza poblacional es desconocida), aplicaremos una prueba $t$ de Student para una muestra.

**Planteamiento de Hipótesis:**

**Hipótesis Nula ($H_0$):** La media de la remuneración mensual unificada es igual al valor de referencia.$$H_0: \mu = 900.00$$

**Hipótesis Alterna ($H_1$):** La media de la remuneración mensual unificada es diferente al valor de referencia.$$H_1: \mu \neq 900.00$$

**Criterio de Decisión:** Se rechaza $H_0$ si el valor-p (p-value) es menor al nivel de significancia establecido ($\alpha = 0.05$).

In [13]:
import pandas as pd
import scipy.stats as stats

# 1. Cargar el conjunto de datos directamente desde el archivo de Excel (.xlsx)
nombre_archivo = '3._remuneraciones_ingresos_adicionales.xlsx'
df = pd.read_excel(nombre_archivo)

# Limpieza automática de espacios en blanco en los títulos de las columnas
df.columns = df.columns.str.strip()

# Buscamos la columna de remuneración por aproximación por si tiene caracteres especiales o tildes
col_remuneracion = [c for c in df.columns if 'Remuneración mensual' in c][0]

# 2. Limpieza rápida: Asegurar que la remuneración sea un valor numérico
df[col_remuneracion] = pd.to_numeric(df[col_remuneracion], errors='coerce')
df = df.dropna(subset=[col_remuneracion])

# 3. Definición del parámetro crítico (Valor de referencia histórico)
mu_referencia = 900.00

# 4. Ejecución del test paramétrico T de Student Unimuestral
t_stat, p_valor = stats.ttest_1samp(df[col_remuneracion], popmean=mu_referencia)

# 5. Cálculos dinámicos adicionales para las impresiones
grados_libertad = len(df[col_remuneracion]) - 1
media_calculada = df[col_remuneracion].mean()

# 6. Visualización formal de resultados (Muestra el 2882 en pantalla)
print("=" * 50)
print(f"Estadístico t obtenido: {t_stat:.4f}")
print(f"Grados de Libertad (df): {grados_libertad}")
print(f"Valor-p (p-value): {p_valor:.4e}")
print("=" * 50)

# 7. Justificación estadística de la decisión
alpha = 0.05
if p_valor < alpha:
    print(f"Decisión: Rechazar H0. Existe evidencia estadística significativa (p < {alpha})")
    print(f"para afirmar que la remuneración media actual en el Municipio de Loja (Mean={media_calculada:.2f})")
    print(f"es diferente del parámetro de referencia de ${mu_referencia:.2f}.")
else:
    print(f"Decisión: No rechazar H0. No existe evidencia estadística suficiente (p >= {alpha})")
    print(f"para afirmar que la remuneración media difiere de ${mu_referencia:.2f}.")

Estadístico t obtenido: -37.7119
Grados de Libertad (df): 2882
Valor-p (p-value): 2.4748e-253
Decisión: Rechazar H0. Existe evidencia estadística significativa (p < 0.05)
para afirmar que la remuneración media actual en el Municipio de Loja (Mean=713.81)
es diferente del parámetro de referencia de $900.00.


**Interpretación y Justificación de la Decisión:**

Dado que el valor-p obtenido ($p = 2.47 \times 10^{-253}$) es estrictamente menor que el nivel de significancia estipulado ($\alpha = 0.05$), se rechaza la hipótesis nula ($H_0$) y se acepta la hipótesis alterna ($H_1$).

**Justificación:**

Existe evidencia estadística altamente significativa y concluyente para afirmar que la remuneración unificada promedio de los servidores del GAD Municipal de Loja ($\mu = \$713.81$) no es igual al parámetro de referencia histórico de $\$900.00$. El valor negativo de nuestro estadístico de prueba ($t = -37.7119$) demuestra empíricamente que el ingreso medio institucional se sitúa muy por debajo de la media nacional planteada en la hipótesis.

**2. Comparación de Grupos (ANOVA y Post-Hoc de Tukey)**

**Análisis de Varianza (ANOVA de 1 factor) por Régimen Laboral**

Evaluaremos si existen diferencias significativas en la Remuneración mensual unificada promedio entre los distintos subgrupos definidos por el Régimen laboral al que pertenece (ej. LOSEP vs. CÓDIGO DE TRABAJO).

**Planteamiento de Hipótesis para ANOVA:**

**Hipótesis Nula ($H_0$):** Las medias de remuneración son iguales en todos los regímenes laborales.

$$H_0: \mu_{\text{LOSEP}} = \mu_{\text{Código de Trabajo}} = \dots = \mu_{k}$$

**Hipótesis Alterna ($H_1$):** Al menos un régimen laboral presenta una media de remuneración mensual distinta a los demás.

$$H_1: \exists \, i, j \quad \text{tal que} \quad \mu_i \neq \mu_j$$

Si el ANOVA resulta estadísticamente significativo ($p < \alpha$), se procederá con la Prueba Post-Hoc de Tukey HSD para realizar comparaciones múltiples y determinar de manera específica qué pares de grupos presentan diferencias significativas controlando la tasa de error familiar mediante el cálculo del rango estudentizado:

$$q = \frac{\bar{y}_{\text{max}} - \bar{y}_{\text{min}}}{\sqrt{MS_{\text{error}} / n}}$$

In [14]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Limpieza de nombres de columnas primarias
df.columns = df.columns.str.strip()

# 2. Copia de seguridad y renombrado estructurado de variables
df_limpio = df.copy()
df_limpio = df_limpio.rename(columns={
    'Régimen laboral al que pertenece': 'Regimen_Laboral',
    'Remuneración mensual unificada': 'Remuneracion'
})

# 3. Limpieza de los nombres de los subgrupos (Alineado con el guion del video)
df_limpio['Regimen_Laboral'] = df_limpio['Regimen_Laboral'].astype(str).str.strip()
df_limpio['Regimen_Laboral'] = df_limpio['Regimen_Laboral'].str.replace('LOSEP-CONCEJAL', 'CONCEJAL', regex=False)
df_limpio['Regimen_Laboral'] = df_limpio['Regimen_Laboral'].str.replace('LOSEP-SOC', 'LOSEP-ADMINISTRATIVO', regex=False)

# Asegurar limpieza física de datos nulos en el subset de análisis
df_limpio = df_limpio.dropna(subset=['Regimen_Laboral', 'Remuneracion'])

# 4. Construcción del modelo lineal y ejecución de la tabla ANOVA de 1 factor
formula_modelo = 'Remuneracion ~ C(Regimen_Laboral)'
modelo_regimen = ols(formula_modelo, data=df_limpio).fit()
tabla_anova = sm.stats.anova_lm(modelo_regimen, typ=2)

print("=" * 20 + " TABLA ANOVA DE 1 FACTOR " + "=" * 20)
print(tabla_anova)
print("=" * 65)

# 5. Evaluación formal y paso a la prueba de comparaciones múltiples de Tukey HSD
p_anova = tabla_anova['PR(>F)'].iloc[0]

if p_anova < 0.05:
    print("\n[RESULTADO]: El ANOVA es estadísticamente significativo (p < 0.05).")
    print("Se ejecuta la Prueba Post-Hoc de Tukey HSD para identificar diferencias por parejas:\n")

    tukey_resultados = pairwise_tukeyhsd(
        endog=df_limpio['Remuneracion'],
        groups=df_limpio['Regimen_Laboral'],
        alpha=0.05
    )
    print(tukey_resultados)
else:
    print("\n[RESULTADO]: El ANOVA NO es significativo (p >= 0.05).")
    print("No existe evidencia estadística para rechazar la homogeneidad de medias.")

==================== TABLA ANOVA DE 1 FACTOR ====================
                          sum_sq      df           F         PR(>F)
C(Regimen_Laboral)  6.389375e+07     3.0  442.264995  2.694739e-236
Residual            1.386425e+08  2879.0         NaN            NaN

[RESULTADO]: El ANOVA es estadísticamente significativo (p < 0.05).
Se ejecuta la Prueba Post-Hoc de Tukey HSD para identificar diferencias por parejas:

                Multiple Comparison of Means - Tukey HSD, FWER=0.05                 
      group1             group2         meandiff  p-adj   lower      upper    reject
------------------------------------------------------------------------------------
CODIGO DE TRABAJO             CONCEJAL  1495.8505   0.0  1325.0565  1666.6444   True
CODIGO DE TRABAJO                LOSEP   252.5691   0.0   230.9689   274.1694   True
CODIGO DE TRABAJO LOSEP-ADMINISTRATIVO   138.1774   0.0    87.8536   188.5012   True
         CONCEJAL                LOSEP -1243.2813   0.0 -1414.016

Interpretación del Análisis de Varianza:

A través del Análisis de Varianza (ANOVA de un factor), se determinó un efecto global crítico del tipo de régimen sobre los ingresos de los empleados ($F(3, 2879) = 442.26, p < 0.05$). Al ser el valor-p obtenido ($2.69 \times 10^{-236}$) marcadamente inferior al umbral de riesgo $\alpha = 0.05$, se rechaza la hipótesis nula de igualdad de medias.Se concluye estadísticamente que el Régimen laboral al que pertenece el servidor influye de manera directa y determinante en la magnitud de su remuneración mensual, certificando que al menos un par de grupos poseen discrepancias significativas en sus medias.